# Transformacion y Clasificacion KNN con Excel

**Autor:** Daniel Antonio Guanga Gallardo  
**Carrera:** Computacion  
**Materia:** Inteligencia Artificial

Este notebook muestra, paso a paso, como construir un flujo completo de Machine Learning usando un archivo Excel pequeno (`dataIA.xlsx`).

## Objetivo
Predecir el nivel de satisfaccion (`nivel_satisfaccion`) a partir de variables de entrada (`sexo`, `edad`, `pais`) usando un modelo KNN (K-Nearest Neighbors).

## Que aprenderas en este notebook
1. Como limpiar y estandarizar texto y nombres de columnas.
2. Como transformar variables categoricas y numericas con `ColumnTransformer`.
3. Como entrenar y evaluar un modelo con validacion Leave-One-Out.
4. Como guardar el modelo y exportar los datos transformados.
5. Como hacer predicciones sobre nuevos registros.

## 1. Importaciones y rutas

En esta seccion se cargan las librerias necesarias y se definen rutas de trabajo.

- `pandas` y `numpy`: manejo y transformacion de datos.
- `scikit-learn`: preprocesamiento, entrenamiento y evaluacion del modelo.
- `pickle`: guardado del modelo entrenado.
- `Path`: construccion de rutas de forma portable.

Tambien se crean diccionarios para mapear etiquetas de texto a clases numericas (y viceversa).

In [ ]:
from __future__ import annotations

import pickle
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import LeaveOneOut, cross_val_predict, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'dataIA.xlsx'
MODEL_PATH = BASE_DIR / 'modelo_knn_satisfaccion.pkl'
TRANSFORMED_PATH = BASE_DIR / 'dataset_transformado_satisfaccion.csv'

TARGET_MAP = {
    'no me gusta': 0,
    'neutral': 1,
    'neutro': 1,
    'me gusta': 2,
}

TARGET_LABELS = {
    0: 'no me gusta',
    1: 'neutral',
    2: 'me gusta',
}

### Explicacion del bloque anterior (importaciones y configuracion)

1. Se importan modulos de Python estandar (`pickle`, `unicodedata`, `Path`) y librerias de ciencia de datos.
2. Se definen rutas para leer el Excel, guardar el modelo y exportar el CSV transformado.
3. `TARGET_MAP` convierte textos de satisfaccion a clases numericas.
4. `TARGET_LABELS` permite volver de numero a texto para explicar resultados de prediccion.

## 2. Funciones auxiliares

Esta seccion encapsula la logica principal en funciones reutilizables.

- `clean_text`: normaliza texto para evitar problemas de acentos y mayusculas.
- `clean_column_name`: estandariza nombres de columnas.
- `make_one_hot_encoder`: crea `OneHotEncoder` compatible con distintas versiones de scikit-learn.
- `load_data`: carga, limpia y valida el Excel.
- `build_pipeline`: crea el pipeline de preprocesamiento + KNN.
- `save_model`: guarda el modelo entrenado en disco.
- `predict_new_sample`: estima la clase de un nuevo ejemplo y su certeza.

In [ ]:
def clean_text(value: object) -> str:
    text = unicodedata.normalize('NFKD', str(value))
    text = text.encode('ascii', 'ignore').decode('ascii')
    return text.strip().lower()


def clean_column_name(value: object) -> str:
    text = clean_text(value)
    text = text.replace(' ', '_').replace('-', '_')
    return text


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def load_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'No se encontro el archivo Excel: {path}')

    df = pd.read_excel(path)
    df.columns = [clean_column_name(column) for column in df.columns]

    rename_map = {}
    if 'pas' in df.columns and 'pais' not in df.columns:
        rename_map['pas'] = 'pais'
    if 'nivelsatisfaccion' in df.columns:
        rename_map['nivelsatisfaccion'] = 'nivel_satisfaccion'

    if rename_map:
        df = df.rename(columns=rename_map)

    required_columns = {'sexo', 'edad', 'pais', 'nivel_satisfaccion'}
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f'Faltan columnas requeridas en el Excel: {sorted(missing)}')

    df = df.copy()
    df['nivel_satisfaccion'] = df['nivel_satisfaccion'].map(lambda value: TARGET_MAP.get(clean_text(value), np.nan))
    df = df.dropna(subset=['nivel_satisfaccion'])
    df['nivel_satisfaccion'] = df['nivel_satisfaccion'].astype(int)

    df['edad'] = pd.to_numeric(df['edad'], errors='coerce')
    df = df.dropna(subset=['edad'])
    df['edad'] = df['edad'].astype(float)

    return df.reset_index(drop=True)


def build_pipeline() -> Pipeline:
    categorical_features = ['sexo', 'pais']
    numeric_features = ['edad']

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', make_one_hot_encoder()),
    ])

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer, categorical_features),
            ('num', numeric_transformer, numeric_features),
        ],
        remainder='drop',
    )

    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('knn', KNeighborsClassifier(n_neighbors=3, metric='minkowski')),
    ])


def save_model(model: Pipeline, path: Path) -> None:
    with path.open('wb') as handle:
        pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)


def predict_new_sample(model: Pipeline, sexo: str, edad: float, pais: str) -> pd.DataFrame:
    sample = pd.DataFrame([{'sexo': sexo, 'edad': edad, 'pais': pais}])
    pred_class = int(model.predict(sample)[0])
    proba = model.predict_proba(sample)[0]
    confidence = float(np.max(proba))

    return pd.DataFrame([
        {
            'prediccion_numero': pred_class,
            'prediccion_texto': TARGET_LABELS.get(pred_class, 'desconocido'),
            'certeza': round(confidence, 4),
        }
    ])

### Explicacion del bloque anterior (funciones auxiliares)

- `clean_text` y `clean_column_name` evitan errores de codificacion y hacen el dataset mas consistente.
- `load_data` incluye validaciones clave: comprueba columnas esperadas y limpia tipos de dato.
- `build_pipeline` define un flujo reproducible: imputacion, codificacion, escalado y clasificacion.
- `save_model` permite persistir el entrenamiento.
- `predict_new_sample` empaqueta una prediccion completa (clase + certeza) para nuevos datos.

Esta modularidad facilita mantenimiento, pruebas y reutilizacion del codigo.

## 3. Carga y vista del dataset

Aqui se ejecuta `load_data(DATA_PATH)` para:

1. Leer el Excel.
2. Estandarizar nombres de columnas.
3. Corregir posibles problemas de codificacion en `pais`.
4. Convertir `nivel_satisfaccion` a valores numericos.
5. Limpiar filas invalidas (si existieran).

Al final se visualiza el dataset ya limpio y la distribucion de la variable objetivo.

In [ ]:
df = load_data(DATA_PATH)
print('Dataset cargado:')
display(df)

print('Distribucion de la variable objetivo:')
display(df['nivel_satisfaccion'].value_counts().sort_index())

### Como interpretar la salida de carga

- Si el dataframe se muestra correctamente, la lectura de Excel fue exitosa.
- La distribucion de `nivel_satisfaccion` te indica si hay clases desbalanceadas.
- Si faltara una columna o hubiera valores invalidos, el notebook fallaria aqui de forma controlada (validacion temprana).

## 4. Entrenamiento y evaluacion (Leave-One-Out)

En esta etapa se separan variables de entrada (`X`) y salida (`y`), se construye el modelo y se evalua con validacion cruzada Leave-One-Out (LOO).

### Por que Leave-One-Out aqui
Como el dataset es muy pequeno, LOO permite entrenar con casi todos los datos y probar en 1 registro por iteracion. Asi se aprovecha mejor la informacion disponible.

### Metricas mostradas
- `Accuracy promedio (LOO)`: promedio de aciertos en todas las iteraciones.
- `Accuracy global`: aciertos totales entre todas las predicciones.
- `Matriz de confusion`: errores y aciertos por clase.
- `Classification report`: precision, recall y F1 por clase.

In [ ]:
X = df[['sexo', 'edad', 'pais']]
y = df['nivel_satisfaccion']

model = build_pipeline()
cv = LeaveOneOut()

scores = cross_val_score(model, X, y, cv=cv)
predictions = cross_val_predict(model, X, y, cv=cv)

print('Accuracy promedio (LOO):', round(float(scores.mean()), 4))
print('Accuracy global:', round(float(accuracy_score(y, predictions)), 4))
print('Matriz de confusion:')
print(confusion_matrix(y, predictions))
print('Reporte de clasificacion:')
print(classification_report(
    y,
    predictions,
    target_names=[TARGET_LABELS[index] for index in sorted(TARGET_LABELS)],
    zero_division=0
))

### Como interpretar la evaluacion

1. Un `accuracy` cercano a 1.0 indica buen rendimiento en este conjunto.
2. La matriz de confusion muestra en que clases se equivoca el modelo.
3. El reporte por clase ayuda a detectar si una etiqueta se predice peor que otras.

Nota: al tener pocos datos, las metricas pueden variar bastante y deben tomarse como orientativas.

## 5. Ajuste final, guardado y exportacion

Despues de evaluar, se entrena el modelo final con todos los datos (`model.fit(X, y)`) para dejarlo listo para uso real.

Ademas:
- Se guarda el modelo en `modelo_knn_satisfaccion.pkl`.
- Se transforma `X` con el preprocesador.
- Se exporta el dataset transformado a `dataset_transformado_satisfaccion.csv`.

Esto permite reutilizar el modelo sin volver a entrenar y auditar como quedaron las variables procesadas.

In [ ]:
model.fit(X, y)
save_model(model, MODEL_PATH)
print('Modelo guardado en:', MODEL_PATH)

transformed = model.named_steps['preprocessor'].transform(X)
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
transformed_df = pd.DataFrame(transformed, columns=feature_names)
transformed_df['nivel_satisfaccion'] = y.to_numpy()
transformed_df.to_csv(TRANSFORMED_PATH, index=False, encoding='utf-8-sig')
print('Dataset transformado guardado en:', TRANSFORMED_PATH)
display(transformed_df.head())

### Resultado del guardado y exportacion

- El archivo `.pkl` contiene el pipeline completo (preprocesamiento + modelo).
- El `.csv` transformado te deja ver exactamente que variables genera `OneHotEncoder` y como queda `edad` tras `StandardScaler`.

Esto es util para trazabilidad del modelo y para compartir insumos con otros analistas.

## 6. Prediccion de un nuevo registro

Aqui se prueba el modelo con un ejemplo manual (`sexo='F', edad=30, pais='Chile'`).

La salida incluye:
- `prediccion_numero`: clase numerica estimada.
- `prediccion_texto`: etiqueta legible de la clase.
- `certeza`: probabilidad asociada a la clase predicha (valor maximo de `predict_proba`).

Puedes cambiar estos valores para simular nuevos casos.

In [ ]:
pred_ejemplo = predict_new_sample(model, sexo='F', edad=30, pais='Chile')
display(pred_ejemplo)

### Lectura de la prediccion final

- `prediccion_numero`: salida codificada del modelo.
- `prediccion_texto`: version comprensible para usuario final.
- `certeza`: confianza del modelo en esa prediccion.

Si quieres hacer pruebas adicionales, cambia `sexo`, `edad` y `pais` en la celda anterior y vuelve a ejecutar.